# Study 6

Compared range estimates with accurate, optimistic, and conservative point estimates to test whether ranges outperform simply promising a later delivery.

In [1]:
# Import packages and locate the data folder.
from pathlib import Path
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import binomtest, t
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/Study 1.xlsx').exists())
os.environ['MPLCONFIGDIR'] = str(ROOT / '.cache/matplotlib')
ITEMS = ['sb_like', 'sb_info', 'sb_useful', 'sb_accurate', 'sb_trust']
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

In [2]:
# Load data, apply exclusions, and calculate satisfaction.
d = pd.read_excel(ROOT / 'data/raw/Study 6.xlsx')
raw_n = len(d)

d = d.loc[(d.ExcludeDuplicateIP != 1) & (d.ExcludeDuplicateID != 1)].copy()
assert len(d) == 801
d['satisfaction'] = d[ITEMS].mean(axis=1)
rows, descriptions = [], []
display(pd.DataFrame([{'raw_n':raw_n, 'replicated_n':len(d), 'paper_n':801, 'match':len(d)==801}]))

,raw_n,replicated_n,paper_n,match
0,803,801,801,True


In [3]:
# Define helpers for regressions and paper comparisons.
def record(study, label, fit, term, target_b=None, target_se=None, sign=1):
    ci = np.sort(np.asarray(fit.conf_int().loc[term]) * sign)
    b, se = (sign * fit.params[term], fit.bse[term])
    row = {'study': study, 'analysis': label, 'n': int(fit.nobs), 'b': b, 'se': se, 'p': fit.pvalues[term], 'ci_low': ci[0], 'ci_high': ci[1], 'published_b': target_b, 'published_se': target_se, 'b_matches_2dp': None if target_b is None else abs(b - target_b) <= 0.0050001, 'se_matches_2dp': None if target_se is None else abs(se - target_se) <= 0.0050001}
    rows.append(row)

def ols(d, y, terms):
    return sm.OLS(d[y], sm.add_constant(d[terms].astype(float)), missing='drop').fit()

def contrast(d, study, labels, treatment, reference, target_b, target_se, label=None, covariates=()):
    frame = d.copy()
    for name, code in labels.items():
        frame[name] = (frame.Condition == code).astype(float)
    terms = [name for name in labels if name != reference] + list(covariates)
    record(study, label or f'{treatment} - {reference}', ols(frame, 'satisfaction', terms), treatment, target_b, target_se)

def describe(d, study, group='all'):
    for cond, row in d.groupby('Condition').satisfaction.agg(['count', 'mean', 'std', 'sem']).iterrows():
        descriptions.append({'study': study, 'group': group, 'condition': cond, 'n': int(row['count']), 'mean': row['mean'], 'sd': row['std'], 'se_mean': row['sem'], 'ci_low': row['mean'] - t.ppf(0.975, row['count'] - 1) * row['sem'], 'ci_high': row['mean'] + t.ppf(0.975, row['count'] - 1) * row['sem']})

In [4]:
# Compare range estimates with the three types of point estimates.
describe(d, 6)
labels = {'optimistic': 1, 'point': 2, 'conservative': 3, 'range': 4}
for a, b, beta, se in [('range', 'point', 0.87, 0.14), ('range', 'conservative', 0.67, 0.14), ('optimistic', 'point', -1.47, 0.14), ('optimistic', 'conservative', -1.68, 0.14), ('conservative', 'point', 0.2, 0.14)]:
    contrast(d, 6, labels, a, b, beta, se)

In [5]:
# Display estimates beside the paper and check agreement at reported precision.
results = pd.DataFrame(rows)
checked = results.published_b.notna()
assert results.loc[checked, 'b_matches_2dp'].all()
assert results.loc[checked, 'se_matches_2dp'].all()
display(results[['analysis','n','b','published_b','se','published_se','p','b_matches_2dp','se_matches_2dp']])

,analysis,n,b,published_b,se,published_se,p,b_matches_2dp,se_matches_2dp
0,range - point,801,0.8677,0.87,0.1427,0.14,1.8591e-09,True,True
1,range - conservative,801,0.6659,0.67,0.1444,0.14,4.6396e-06,True,True
2,optimistic - point,801,-1.4747,-1.47,0.1424,0.14,1.1218e-23,True,True
3,optimistic - conservative,801,-1.6765,-1.68,0.1440,0.14,4.9848e-29,True,True
4,conservative - point,801,0.2018,0.20,0.1447,0.14,1.6359e-01,True,True


In [6]:
# Enter the published means and standard deviations for comparison.
published = {1:(3.05,1.66),2:(4.53,1.51),3:(4.73,1.41),4:(5.40,1.10)}

In [7]:
# Compare group means and standard deviations with the published table.
table = d.groupby('Condition').satisfaction.agg(n='count',mean='mean',sd='std')
table['condition'] = table.index.map({v:k for k,v in labels.items()})
table['paper_mean'] = [published[c][0] for c in table.index]
table['paper_sd'] = [published[c][1] for c in table.index]
table['match'] = ((table['mean']-table.paper_mean).abs()<=.0050001)&((table.sd-table.paper_sd).abs()<=.0050001)
assert table['match'].all()
display(table)

,n,mean,sd,condition,paper_mean,paper_sd,match
Condition,,,,,,,
1,205,3.0527,1.6569,optimistic,3.05,1.66,True
2,201,4.5274,1.5088,point,4.53,1.51,True
3,192,4.7292,1.4132,conservative,4.73,1.41,True
4,203,5.3951,1.0957,range,5.40,1.10,True
